In [1]:
import ee
import geemap
import geehydro
import geopandas as gpd
import json
import os

In [2]:
ee.Initialize()

In [3]:
def obtem_ano(ano):
    # Carrega o shapefile e filtra para o PARNA Serra da Canastra
    pnsc = gpd.read_file(r"G:\Meu Drive\@EquipeGEO\zz.Bases\ICMBio\UC_Fed_nov_2020.shp")
    pnsc = pnsc.loc[pnsc["nome"] == "PARQUE NACIONAL DA SERRA DA CANASTRA"]
    pnsc_geojson = pnsc.to_json()

    # Converte o shapefile para um objeto Earth Engine
    pnsc_ee = ee.FeatureCollection(json.loads(pnsc_geojson))

    # Função para selecionar a coleção Landsat com base no ano
    def selecionarColecaoLandsat(ano):
        if ano >= 1984 and ano <= 1999:
            return "LANDSAT/LT05/C02/T1_L2"  # Landsat 5
        elif ano >= 1999 and ano <= 2012:
            return "LANDSAT/LE07/C02/T1_L2"  # Landsat 7
        elif ano >= 2013:
            return "LANDSAT/LC08/C02/T1_L2"  # Landsat 8
        else:
            raise ValueError("Ano fora do intervalo disponível para coleções Landsat")

    # Função para mascarar nuvens e sombras
    def maskLandsat(image):
        qa = image.select('QA_PIXEL')
        cloud = qa.bitwiseAnd(1 << 3).eq(0)
        shadow = qa.bitwiseAnd(1 << 5).eq(0)
        mask = cloud.And(shadow)
        return image.updateMask(mask)

    # Parâmetros de entrada
    data_inicio = f"{ano}-07-01"
    data_fim = f"{ano}-10-31"

    # Seleciona a coleção com base no ano
    colecao = selecionarColecaoLandsat(ano)

    # Filtra e aplica a máscara na coleção de imagens
    dataset = ee.ImageCollection(colecao) \
        .filterDate(data_inicio, data_fim) \
        .filterBounds(pnsc_ee) \
        .map(maskLandsat)

    # Seleciona as bandas para a visualização True Color e NBR
    if ano <= 2012:  # Landsat 5 e 7
        trueColor432 = dataset.select(["SR_B3", "SR_B2", "SR_B1"])
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B4", "SR_B7"]).rename("NBR"))
    else:  # Landsat 8
        trueColor432 = dataset.select(["SR_B4", "SR_B3", "SR_B2"])
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B5", "SR_B7"]).rename("NBR"))

    trueColor432Vis = {
        "min": 0.0,
        "max": 0.4,
    }

    # Obtém o NBR mínimo (severidade máxima)
    nbrMin = nbr.min()

    # Classifica a severidade das queimadas com base em novos limiares
    classificacao = nbrMin.expression(
        "(nbr < -0.5) ? 3 : "  # Queimada de alta intensidade (vermelho)
        "((nbr >= -0.5) && (nbr < -0.2)) ? 2 : "  # Queimada de moderada intensidade (amarelo)
        "((nbr >= -0.2) && (nbr < -0.1)) ? 1 : "  # Queimada de baixa intensidade (laranja)
        "0",  # Área não queimada (não exibida)
        {"nbr": nbrMin}
    )

    # Parâmetros de visualização para a classificação de severidade
    classificacaoVis = {
        "min": 1,
        "max": 3,
        "palette": ["orange", "yellow", "red"]  # Baixa intensidade (laranja), Moderada (amarelo), Alta (vermelho)
    }

    # Calcula o centróide do PNSC
    centroid = pnsc_ee.geometry().centroid()

    # Inicializa o mapa centrado no centróide do PNSC
    Map = geemap.Map(center=(centroid.coordinates().get(1).getInfo(), centroid.coordinates().get(0).getInfo()), zoom=10)

    # Adiciona camadas ao mapa
    Map.addLayer(trueColor432.median(), trueColor432Vis, "True Color (432)")
    Map.addLayer(nbrMin, {"min": -1, "max": 1}, "NBR Mínimo (Maior Severidade de Queimada)")
    Map.addLayer(classificacao.updateMask(classificacao.gt(0)), classificacaoVis, f"Class Sever {ano}")
    Map.addLayer(ee.Image().paint(pnsc_ee, 0, 2), {}, "PARNA Serra da Canastra")

    # Retorna o mapa atualizado
    return Map

In [4]:
mapa = obtem_ano(1984)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [5]:
mapa = obtem_ano(1985)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [10]:
mapa = obtem_ano(1986)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [11]:
mapa = obtem_ano(1987)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [12]:
mapa = obtem_ano(1988)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [13]:
mapa = obtem_ano(1989)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [14]:
mapa = obtem_ano(1990)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [15]:
mapa = obtem_ano(2000)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [6]:
mapa = obtem_ano(2001)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [17]:
mapa = obtem_ano(2002)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369024, -46.5843309803493], controls=(WidgetControl(options=['position', 'transparent…

In [18]:
mapa = obtem_ano(2003)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [19]:
mapa = obtem_ano(2004)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [20]:
mapa = obtem_ano(2005)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [21]:
mapa = obtem_ano(2006)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [22]:
mapa = obtem_ano(2007)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369024, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [23]:
mapa = obtem_ano(2008)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [24]:
mapa = obtem_ano(2009)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [7]:
mapa = obtem_ano(2010)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [8]:
mapa = obtem_ano(2011)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [27]:
mapa = obtem_ano(2012)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369024, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [28]:
mapa = obtem_ano(2013)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [29]:
mapa = obtem_ano(2014)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [30]:
mapa = obtem_ano(2015)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [9]:
mapa = obtem_ano(2016)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [32]:
mapa = obtem_ano(2017)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [33]:
mapa = obtem_ano(2018)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [34]:
mapa = obtem_ano(2019)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [35]:
mapa = obtem_ano(2020)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [36]:
mapa = obtem_ano(2021)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [37]:
mapa = obtem_ano(2022)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [38]:
mapa = obtem_ano(2023)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [39]:
mapa = obtem_ano(2024)
mapa.setControlVisibility(layerControl=True, fullscreenControl=True, latLngPopup=True)
mapa

Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [44]:
!jupyter nbconvert --to html nova_tentativa copy 2.ipynb

[NbConvertApp] WARNING | pattern 'copy' matched no files
[NbConvertApp] WARNING | pattern '2.ipynb' matched no files
[NbConvertApp] Converting notebook nova_tentativa.ipynb to html
[NbConvertApp] Writing 323230 bytes to nova_tentativa.html
